# Uczenie Maszynowe — Laboratorium 8


## 1. Analiza oddanych prac — wybór własnej listy

Do porównania wybrałem **Listę z klasyfikacją danych niezrównoważonych** (predykcja "matchu" na danych Speed Dating). Wybór uzasadniam tym, że:

- to zadanie, w którym świadomie zastosowałem najwięcej różnych technik (resampling, ważenie klas, kalibracja progu, własna macierz kosztów) — najlepiej oddaje mój sposób pracy z problemem klasyfikacji binarnej z rzadką klasą pozytywną,
- problem (match = zdarzenie rzadkie) jest strukturalnie tego samego typu, co duża klasa rzeczywistych problemów biznesowych z silnym niezbalansowaniem (fraud, churn, awarie) — łatwo znaleźć dla niego sensowny odpowiednik konkursowy.


## 2. Wybór pracy konkursowej

**Wybrany konkurs:** IEEE-CIS Fraud Detection(https://www.kaggle.com/competitions/ieee-fraud-detection)

**Uzasadnienie doboru:** to ten sam typ problemu ML — **binarna klasyfikacja z silnie niezrównoważonymi klasami** (fraud jest ok. 3,5% obserwacji, podobnie jak match w danych Speed Dating jest zdarzeniem rzadkim). W obu przypadkach kluczowe są te same techniki: resampling, ważenie klas, dobór progu decyzyjnego, oraz budowanie cech na bazie identyfikatora "podmiotu" (user_id / card_id), a nie tylko pojedynczej obserwacji.

Ponieważ konkurs ma już zakończoną edycję z bogatą sekcją rozwiązań, mogłem wskazać konkretne prace zamiast tylko opisów tekstowych. Dostępne notatniki były niestety rozwiązań troche gorszych (brązowych) złote i srebrne prace nie zostały szczegółowo udostępnione jedynie jakieś ogulne opisy. 

| Kategoria | Wybrana praca | Dlaczego ta |
|---|---|---|
| EDA | ieee-cis-fraud-detection-model-ml.ipynb (notebook "brązowy") | Jedyna w pełni dostępna praca z kodem, ma rozbudowaną, wieloetapową sekcję EDA (rozkłady, korelacje, testy statystyczne, analiza braków danych per klasa) |
| Preprocessing | ieee-cis-fraud-detection-model-ml.ipynb + opis rozwiązania złotego zespołu (Giba/Tony/sggpls/krivoship) | Notebook brązowy pokazuje konkretny kod (target/frequency encoding, optymalizacja pamięci); opis złotego zespołu pokazuje *koncepcyjnie* bardziej zaawansowany krok (agregaty po UID), którego kodu niestety nie udostępniono |
| Optymalizacja hiperparametrów | Chris Deotte, *"XGB Fraud with Magic"* (kaggle.com/code/cdeotte/xgb-fraud-with-magic-0-9600) oraz projekt *Bank Transfer Fraud Detection* (Optuna + LightGBM) | Notebook brązowy zawiera modele trenowane z parametrami domyślnymi — żeby ocenić tę kategorię uczciwie, sięgnąłem po inne publicznie dostępne rozwiązanie z tego konkursu, które faktycznie tunuje hiperparametry |
| Uczenie modelu | ieee-cis-fraud-detection-model-ml.ipynb + opisy złotego/srebrnego zespołu | Notebook brązowy pokazuje kod treningu 8 modeli równolegle; opisy zespołów pokazują kluczowy, niewidoczny w kodzie element — blending modeli i walidację czasową |
| Ewaluacja | ieee-cis-fraud-detection-model-ml.ipynb | Najbardziej rozbudowana, kodowa sekcja ewaluacji ze wszystkich dostępnych materiałów (learning curves, threshold tuning, confusion matrix, zapis metadanych) |

Najlepsze miejsca opublikowały na forum konkursu tylko opisy tekstowe rozwiązania, bez kodu.  Dlatego część porównań opiera się na opisie koncepcyjnym.

---

## 3. Porównanie do pracy konkursowej

### 3.1 Eksploracyjna analiza danych (EDA)

**Czym się różnią etapy:**
Mój EDA (Speed Dating) ogranicza się do rozkładu zmiennej docelowej, kilku wykresów KDE/boxplot i jednej macierzy korelacji cech samooceny. Notebook brązowy robi EDA wielowarstwowo: rozkład klasy, rozkład kwoty transakcji per klasa, fraud rate per kategoria (typ karty, domena e-mail), cechy czasowe (godzina/dzień tygodnia vs. fraud rate), korelacja cech V/ID z targetem, test statystyczny dla najbardziej skorelowanych cech, oraz analiza brakujących danych rozłączna per klasa (czy braki danych same w sobie różnią się między fraud/nie-fraud).

**Co najsilniej wpływa na wyniki:**
Największą różnicą jest właśnie EDA brakujących danych per klasa oraz EDA czasowa — w fraud detection okazuje się, że *pora dnia* i *sam fakt brakującej wartości* są silnym sygnałem, co bez takiej analizy łatwo przeoczyć. W moich danych analogiczny sygnał (np. czy brak odpowiedzi na pewne pytania koreluje z matchem) nie był sprawdzony wcale.

**Jak zastosować to w mojej liście:**
- Dodać sprawdzenie, czy braki danych w konkretnych kolumnach (np. `field`, `income`) różnią się częstością między grupą match=1 i match=0 — to nic nie kosztuje, a może wykryć sygnał.
- Dodać test statystyczny dla cech, które wizualnie wyglądają na różniące grupy, zamiast oceniać "na oko" z wykresu.

---

### 3.2 Preprocessing

**Czym się różnią etapy:**
Mój preprocessing: usunięcie kolumn z wyciekiem (`leakage_columns`), budowa `user_ids` z cech demograficznych, imputacja medianą/stałą wartością, one-hot encoding (`pd.get_dummies`) na całym połączonym zbiorze.

Notebook brązowy: optymalizacja pamięci (redukcja typów danych), **target encoding** (fraud rate per kategoria) i **frequency encoding** połączone w jeden ważony "fraud score", normalizacja MinMax, dopiero potem PCA/SVD do redukcji wymiarowości przed podaniem do modeli.

Opis zespołu złotego idzie jeszcze dalej — buduje **identyfikator użytkownika (UID)** z kombinacji kolumn (`card1+addr1+dzień-D1+P_emaildomain`), a następnie generuje z niego ok. 80 cech agregujących (`count`, `mean`, `std`, `median` czasu między transakcjami i kwoty) per UID.

**Co najsilniej wpływa na wyniki:**
Z opisu wynika jednoznacznie: **agregaty zbudowane na UID dały największy pojedynczy skok wyniku** w całym rozwiązaniu. To silniejszy czynnik niż sam wybór metody encodingu czy redukcji wymiarowości.

**Jak zastosować to w mojej liście:**
- Mam już `user_ids` — ale używam go tylko do podziału train/test, nie do budowy cech. Mogę policzyć analogiczne agregaty: `user_match_rate` (tylko z train, żeby nie przeciekł target), `user_n_interactions`, `user_avg_attractive_given`, odchylenie cechy danej osoby od średniej w jej grupie.
- Zamienić `pd.get_dummies` na target/frequency encoding dla cech wysokiej kardynalności (np. `field`), zamiast rozbijać je na dziesiątki kolumn 0/1.

---

### 3.3 Optymalizacja hiperparametrów

**Czym się różnią etapy:**
W mojej liście HPO jest realnie zrobione — pełny grid search po `depth`, `learning_rate`, `l2_leaf_reg` dla CatBoost, połączony jednocześnie z szukaniem optymalnego progu decyzyjnego.

W notebooku brązowym **HPO praktycznie nie istnieje** — 8 modeli (LogReg, KNN, drzewo, RF, GradientBoosting, XGBoost, LightGBM, CatBoost) jest trenowanych z parametrami w większości domyślnymi lub jedną odgórnie wybraną liczbą `n_estimators=100`. To realna słabość tego rozwiązania, mimo że poza tym jest solidne.

Dlatego do tej kategorii sięgnąłem po inny publiczny projekt rozwiązujący ten sam konkurs, w którym hiperparametry LightGBM faktycznie tunowano metodą Optuna (Bayesian optimization), z konkretnymi znalezionymi wartościami (`num_leaves`, `max_depth`, `learning_rate`, `subsample` itd., walidacja PR-AUC).

**Co najsilniej wpływa na wyniki:**
Tu różnica jest jakościowa, nie ilościowa: mój kod robi **brute-force grid search** (sprawdza wszystkie kombinacje z ustalonej, małej siatki) — co działa, ale słabo skaluje się przy większej liczbie hiperparametrów. Optuna w przykładzie konkursowym robi **przeszukiwanie bayesowskie** (kolejne próby są kierowane wynikami poprzednich, nie losowe/siatkowe) — przy tej samej liczbie prób trafia bliżej optimum.

Druga, ważniejsza różnica: **mój HPO ma błąd metodologiczny** — wybieram najlepsze hiperparametry *i* próg na podstawie wyniku liczonego bezpośrednio na zbiorze testowym (pętla odpytuje X_test/y_test wielokrotnie). To zawyża wynik. Żaden z poważnych rozwiązań konkursowych (ani opis złotego zespołu, który explicite pisze o użyciu CV do HPO, ani przykład z Optuną, który stosuje 3-fold CV podczas optymalizacji) nie robi tego w ten sposób.

**Jak zastosować to w mojej liście:**
- Wydzielić osobny zbiór walidacyjny (train/val/test, nie tylko train/test) — HPO i dobór progu robić na val, test odpalić raz, na końcu.
- Przy większej liczbie hiperparametrów zamienić grid search na losowe przeszukiwanie (RandomizedSearchCV) lub Optuna — szybciej trafia w sensowny obszar przestrzeni parametrów niż pełna siatka.

---

### 3.4 Uczenie modelu

**Czym się różnią etapy:**
Trenuję 4 modele dopasowane pod niezbalansowanie (RF z `class_weight`, BalancedRandomForest, EasyEnsemble, CatBoost z wagami klas) i wybieram jeden najlepszy do dalszej analizy.

Notebook brązowy trenuje 8 generycznych modeli na dwóch wariantach zbalansowanych danych (undersampling, SMOTE) po redukcji wymiarowości (PCA/SVD) — podejście "spróbuj wszystkiego i wybierz najlepsze z rankingu", bez głębszej refleksji nad tym, dlaczego dany model wygrywa.

Opisy złotego i srebrnego zespołu pokazują element, którego nie ma w żadnym z dwóch notebooków z kodem: blending (uśrednianie) kilku modeli wytrenowanych niezależnie jako główny czynnik sukcesu, oraz walidację czasową (foldy odzwierciedlające realny odstęp czasowy train/test), a nie zwykły losowy k-fold.

**Co najsilniej wpływa na wyniki:**
Z opisów wynika to bardzo jednoznacznie: blending dał zespołom więcej niż dobór pojedynczego najlepszego modelu, mimo że poszczególne modele były do siebie podobne (wszyscy używali LightGBM z podobnymi parametrami — różnica leżała w *różnorodności cech wejściowych*, nie w różnorodności algorytmów). To jest odwrotność intuicji "wybierz jeden najlepszy model" — w obu opisach wybór padł na uśrednienie kilku, niekoniecznie najlepszego pojedynczo.

**Jak zastosować to w mojej liście:**
- Zamiast wybierać jeden "najlepszy" model z czterech wytrenowanych, sprawdzić proste lub ważone uśrednienie `predict_proba` z 2–4 modeli i porównać z najlepszym pojedynczym wynikiem.
- Zamienić jednorazowy podział train/test na walidację wielofoldową (k-fold po `user_id`), żeby ocena modeli była stabilniejsza, a nie zależna od jednego losowego podziału.

---

### 3.5 Ewaluacja modelu

**Czym się różnią etapy:**
Moja ewaluacja jest najbardziej rozbudowana biznesowo: ROC/PR-AUC dla 4 modeli, własna macierz kosztów (PLN za TP/FP/FN/TN), porównanie strategii "Romantycznej" (maks. recall) vs "Biznesowej" (maks. zysk), metryki rekomendacyjne specyficzne dla domeny (Precision@K, MRR).

Notebook brązowy ma bardziej rozbudowaną ewaluację techniczną: learning curves dla wszystkich 8 modeli (diagnoza under/overfitting per model), pełne przeszukanie progu decyzyjnego z wykresem Precision/Recall/F1 w funkcji progu, oraz zapis modelu, reduktora wymiarowości i metadanych progu do plików — czyli element powtarzalności/wdrożenia, którego u mnie nie ma.

**Co najsilniej wpływa na wyniki:**
Tu różnica nie dotyczy wyniku modelu, a wiarygodności oceny tego wyniku. Learning curves z notebooka brązowego pozwalają odróżnić "model jest słaby" od "model jest przeuczony" od "potrzeba więcej danych" — u mnie taka diagnostyka nie istnieje, więc nie wiem np. czy EasyEnsemble wypada gorzej, bo jest gorszym modelem, czy bo akurat przy mojej wielkości zbioru nie ma się czego nauczyć.

**Jak zastosować to w mojej liście:**
- Dodać learning_curve (sklearn) dla najlepszych 1-2 modeli — szybki sposób na zdiagnozowanie, czy warto dokładać dane/cechy, czy raczej regularyzować model.
- Zapisać finalnie wybrany model, próg decyzyjny i metadane — to drobny nawyk, ale ułatwia odtworzenie wyniku bez ponownego trenowania wszystkiego od zera.

---

## Podsumowanie

| Obszar | Mój kod robi lepiej | Konkurs robi lepiej | Najsilniejszy czynnik wpływu na wynik |
|---|---|---|---|
| EDA | — | analiza braków per klasa, testy statystyczne, EDA czasowa | wykrycie sygnału w *brakach danych* i czasie |
| Preprocessing | usunięcie wycieku, świadomy split bez przecieku usera | target/frequency encoding, agregaty po UID | **agregaty po UID** (największy pojedynczy skok wyniku w opisach) |
| HPO | realne HPO + dobór progu (ale z błędem metodologicznym) | przeszukiwanie bayesowskie (Optuna), CV przy tuningu | rozdzielenie zbioru do tuningu od zbioru do oceny |
| Uczenie modelu | różnorodność technik dla niezbalansowania | blending modeli, walidacja czasowa | **blending**, nie wybór jednego "najlepszego" modelu |
| Ewaluacja | metryki biznesowe (macierz kosztów, Precision@K, MRR) | learning curves, pełna analiza progu, zapis modelu | diagnostyka over/underfitting przed wyciąganiem wniosków |

---

## Źródła

- Notebook "brązowy": ieee-cis-fraud-detection-model-ml.ipynb 
- Opisy rozwiązań złotego i srebrnego zespołu — wątki dyskusyjne konkursu IEEE-CIS Fraud Detection na Kaggle (materiał własny, wklejony w treści zadania)
- Chris Deotte, *XGB Fraud with Magic* — https://www.kaggle.com/code/cdeotte/xgb-fraud-with-magic-0-9600
- *Bank Transfer Fraud Detection – IEEE-CIS Kaggle Competition* (przykład HPO z Optuna) — https://github.com/vdafeider/ieee_fraud_detection_analysis_and_ml
- *IEEE-CIS Fraud Detection - Top 5% Solution*, Towards Data Science — https://towardsdatascience.com/ieee-cis-fraud-detection-top-5-solution-5488fc66e95f/